# TV3 - Crop classifier EDA (tuần 1)

**Chạy từ đâu:** mở notebook trong thư mục `notebooks/`, chọn kernel Python 3.12 đã cài `requirements.txt`.
Cell đầu tự tìm thư mục gốc repo nên không phụ thuộc thư mục đang mở của Jupyter.

**Dữ liệu cần có:**
- `data/processed/v1/train.jsonl` (manifest v1 của TV1, đã có trong repo) - nguồn duy nhất để chọn ảnh xem trực quan.
- `data/raw/swg_camera_traps.bounding_boxes.with_species.zip` - tải bằng `scripts/download_metadata.py`; chỉ dùng cho thống kê metadata (không mở ảnh).

Logic nằm trong `scripts/tv3_crop_eda.py`, có test tại `tests/test_tv3_crop_eda.py`.

In [ ]:
import json, sys
from pathlib import Path
from datetime import datetime, timezone

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'prepare_swg.py').is_file())
sys.path.insert(0, str(ROOT / 'scripts'))
import tv3_crop_eda as eda  # dùng valid_bbox của TV1 qua prepare_swg

REPORT_DIR = ROOT / 'reports' / 'tv3'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Repo root:', ROOT)

## Bước 1 - Thống kê box trên toàn bộ metadata box gốc

Phạm vi: **toàn bộ annotation trong file box metadata gốc** (mọi split, mọi loài). Đây chỉ là kiểm tra metadata,
không tải hay mở ảnh nào. Đọc bằng `ijson` với `use_float=True` để tọa độ thập phân là `float`, rồi áp dụng `valid_bbox` của TV1.

In [ ]:
metadata_scan = eda.scan_box_zip(eda.BOX_ZIP)
print(json.dumps(metadata_scan, indent=2))

## Bước 2 - Đọc train manifest

Chỉ đọc `train.jsonl`. Box dùng để vẽ là `boxes[].bbox_xywh` (pixel, đã làm sạch bởi TV1).

In [ ]:
train = eda.load_manifest(eda.TRAIN_MANIFEST)
manifest_info = {
    'path': eda.TRAIN_MANIFEST.relative_to(ROOT).as_posix(),
    'sha256': eda.file_sha256(eda.TRAIN_MANIFEST),
    'git_commit': eda.git_commit_of(eda.TRAIN_MANIFEST),
    'records': len(train),
}
assert all(r['split'] == 'train' for r in train)
print(manifest_info)

## Bước 3 - Lấy mẫu 10 ảnh/loài từ train và lưu hình

Seed 42; xoay vòng qua các location đã xáo trộn để đa dạng bối cảnh. Nếu loài có ít hơn 10 ảnh hợp lệ thì báo đúng
số thực tế, không lấy bù từ val/test.

Đầu ra mỗi loài trong `reports/tv3/<loài>/`:
- `NN_<id>_bbox.jpg` - ảnh có vẽ box (được commit)
- `grid_<loài>.png` - lưới tổng hợp (được commit)
- `NN_<id>_raw.jpg` - ảnh gốc, **chỉ lưu cục bộ**, bị `.gitignore` bỏ qua

In [ ]:
audit, errors, summary = [], [], {}
seen_hashes = {}
for species in eda.TARGET_SPECIES:
    rows, errs, counts = eda.process_species(species, train, REPORT_DIR / species, seen_hashes=seen_hashes)
    audit += rows
    errors += errs
    summary[species] = counts
    print(f"{species:26s} chọn {counts['selected']:2d} | lưu thành công {counts['saved']:2d} | lỗi {counts['failed']}")

not_train = eda.ids_not_in_train(audit, train)
assert not not_train, f'Có ID không thuộc train manifest: {not_train}'
assert len({r['image_id'] for r in audit}) == len(audit), 'Có ID bị chọn trùng'
print('Tất cả', len(audit), 'ID đã chọn đều thuộc train manifest và không trùng.')

## Bước 4 - Lưu audit và biên bản lỗi

In [ ]:
generated = datetime.now(timezone.utc).isoformat(timespec='seconds')
(REPORT_DIR / 'sampled_audit.json').write_text(json.dumps({
    'generated_at': generated,
    'seed': eda.SEED,
    'manifest': manifest_info,
    'method': 'Train-only; round-robin over shuffled locations per species; random.Random(f"{seed}:{species}")',
    'samples': audit,
}, indent=2, ensure_ascii=False), encoding='utf-8')

(REPORT_DIR / 'bien_ban_loi_w1.json').write_text(json.dumps({
    'generated_at': generated,
    'metadata_scan': metadata_scan,
    'sample_download': {
        'per_species': summary,
        'selected': sum(c['selected'] for c in summary.values()),
        'saved': sum(c['saved'] for c in summary.values()),
        'download_failed': sum(e['type'] == 'download_failed' for e in errors),
        'decode_failed': sum(e['type'] == 'decode_failed' for e in errors),
        'duplicate_bytes_in_sample': sum(e['type'] == 'duplicate_bytes_in_sample' for e in errors),
        'errors': errors,
    },
    'limitations': [
        'SHA-256 chỉ so trùng byte giữa các ảnh mẫu đã tải trong lần chạy này; không kết luận toàn dataset hay các split sạch trùng nội dung.',
        'Thống kê metadata_scan bao phủ toàn bộ file box gốc; phần mẫu ảnh chỉ thuộc train manifest của 8 lớp.',
    ],
}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Đã lưu reports/tv3/sampled_audit.json và reports/tv3/bien_ban_loi_w1.json')